# Assistente de Voz Inteligente

Projeto desenvolvido durante o bootcamp DIO.

Um assistente pessoal que entende sua voz e responde de forma natural, combinando as melhores tecnologias de IA disponíveis.

**Tecnologias:** Whisper (OpenAI) + ChatGPT + Google TTS

## 1. Configuração Inicial

Primeiro, vamos instalar as bibliotecas necessárias.

In [ ]:
# Instalação das dependências
!pip install -q openai gTTS
!pip install -q git+https://github.com/openai/whisper.git

## 2. Configurar API Key da OpenAI

Você precisa de uma chave de API da OpenAI. Se ainda não tem:
1. Acesse: https://platform.openai.com/api-keys
2. Crie uma conta (ou faça login)
3. Clique em "+ Create new secret key"
4. Copie a chave gerada

**Nota:** A chave começa com 'sk-'

In [ ]:
import os

# Cole sua API key aqui entre as aspas
OPENAI_API_KEY = "sua_chave_aqui"

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print("API Key configurada com sucesso!")

## 3. Gravação de Áudio

Vamos usar JavaScript para acessar o microfone do seu computador pelo navegador.

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
    const reader = new FileReader()
    reader.onloadend = e => resolve(e.srcElement.result)
    reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true })
    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)
    recorder.start()
    await sleep(time)
    recorder.onstop = async () => {
        blob = new Blob(chunks)
        text = await b2text(blob)
        resolve(text)
    }
    recorder.stop()
})
"""

def gravar_audio(segundos=5):
    """Grava áudio do microfone pelo navegador"""
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (segundos * 1000))
    audio = b64decode(js_result.split(',')[1])
    
    arquivo = '/content/minha_fala.wav'
    with open(arquivo, 'wb') as f:
        f.write(audio)
    
    return arquivo

print("Pronto para gravar!")

## 4. Função Principal do Assistente

Agora vamos criar o assistente completo.

In [ ]:
import whisper
import openai
from gtts import gTTS
import time

class AssistenteVoz:
    """Assistente que ouve, entende e responde em voz"""
    
    def __init__(self, idioma='pt'):
        self.idioma = idioma
        self.modelo_whisper = whisper.load_model("base")
        openai.api_key = os.environ.get('OPENAI_API_KEY')
        print("Assistente iniciado! Pronto para conversar.")
    
    def ouvir(self, arquivo_audio):
        """Converte áudio em texto usando Whisper"""
        print("Transcrevendo o que você disse...")
        resultado = self.modelo_whisper.transcribe(arquivo_audio, language=self.idioma)
        texto = resultado["text"].strip()
        print(f"Você disse: {texto}")
        return texto
    
    def pensar(self, mensagem):
        """Envia para o ChatGPT e recebe resposta"""
        if not mensagem:
            return "Não entendi direito. Pode repetir?"
        
        print("Pensando na resposta...")
        
        resposta = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Você é um assistente amigável e prestativo que conversa de forma natural em português brasileiro."},
                {"role": "user", "content": mensagem}
            ],
            max_tokens=200,
            temperature=0.8
        )
        
        texto_resposta = resposta.choices[0].message.content.strip()
        print(f"Resposta: {texto_resposta}")
        return texto_resposta
    
    def falar(self, texto):
        """Converte texto em áudio e reproduz"""
        if not texto:
            return
        
        print("Gerando áudio da resposta...")
        tts = gTTS(text=texto, lang=self.idioma, slow=False)
        arquivo = "/content/resposta.mp3"
        tts.save(arquivo)
        
        # Reproduz o áudio
        display(Audio(arquivo, autoplay=True))
    
    def conversar(self, segundos=5):
        """Executa uma conversa completa: ouvir -> pensar -> falar"""
        print("\n" + "="*50)
        print("Iniciando conversa...")
        print("="*50 + "\n")
        
        # 1. Grava o áudio
        print("Fale algo (gravando por {} segundos)...".format(segundos))
        arquivo = gravar_audio(segundos)
        display(Audio(arquivo, autoplay=False))  # Mostra o áudio gravado
        
        # 2. Transcreve
        mensagem = self.ouvir(arquivo)
        
        # 3. Processa com ChatGPT
        resposta = self.pensar(mensagem)
        
        # 4. Fala a resposta
        self.falar(resposta)
        
        print("\n" + "="*50)
        print("Conversa finalizada!")
        print("="*50 + "\n")

# Cria o assistente
assistente = AssistenteVoz()
print("\nAssistente criado com sucesso! Use: assistente.conversar()")

## 5. Vamos Conversar!

Execute a célula abaixo e fale algo quando aparecer "Fale algo...".

O assistente vai:
1. Ouvir o que você disser
2. Transcrever para texto
3. Pensar na melhor resposta
4. Responder em voz alta

**Dica:** Se quiser mais tempo para falar, mude o número dentro dos parênteses. Exemplo: `assistente.conversar(10)` para 10 segundos.

In [ ]:
# Inicie a conversa!
assistente.conversar(5)  # Grava por 5 segundos

## 6. Experimente Diferentes Perguntas

Tente perguntar coisas como:
- "Qual a capital do Brasil?"
- "Me conte uma piada"
- "O que é inteligência artificial?"
- "Me dê uma receita de bolo"

Execute a célula acima quantas vezes quiser!

## 7. Modo Texto (sem voz)

Se preferir ver só o texto sem ouvir a resposta em voz:

In [ ]:
# Apenas transcreve e mostra a resposta em texto
print("Gravando...")
arquivo = gravar_audio(5)
texto = assistente.ouvir(arquivo)
resposta = assistente.pensar(texto)

print("\n=== RESULTADO ===")
print(f"Você: {texto}")
print(f"Assistente: {resposta}")
print("=================\n")

---

## Sobre o Projeto

**Autor:** Acib ABBADE
**Bootcamp:** DIO
**Data:** Abril/2026

**Tecnologias utilizadas:**
- OpenAI Whisper (reconhecimento de fala)
- OpenAI GPT-3.5 (processamento de linguagem)
- Google Text-to-Speech (síntese de voz)

**Repositório no GitHub:**
https://github.com/acibabbadecastro/dio-voice-assistant